# Distance to TLS and bronchi

**Pinned Environment:** [`conda_envs/space2_20250604.yml`](../conda_envs/space2_20250604.yml)  

In [ ]:
from pathlib import Path
import sys
import os
import time
import warnings
import re
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata
import scanpy as sc
import squidpy as sq


import scipy
from scipy.spatial import distance_matrix
from scipy.spatial.distance import cdist
from scipy import stats
import scipy.ndimage as ndi
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks

from shapely.geometry import Point

from shapely.geometry import MultiPolygon
from alphashape import alphashape
from alphashape import optimizealpha
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer

import pickle
import joblib

import random
import matplotlib.colors as mcolors

import matplotlib as mpl
mpl.rcParams['axes.titlesize'] = 24
mpl.rcParams['pdf.fonttype'] = 42

import warnings
warnings.simplefilter(action='ignore', category=Warning)

sys.version_info

## Local file info

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[0]))

from config.paths import BASE_OUTDIR, INPUTS_DIR, FUNCTIONS_DIR

inputs_dir = INPUTS_DIR
dist_out_dir = os.path.join(BASE_OUTDIR, "downstream_analysis/distance")


plot_out_dir = os.path.join(dist_out_dir, 'plots')

print(inputs_dir)
print(dist_out_dir)

In [ ]:
import importlib
if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.append(str(FUNCTIONS_DIR))

# import plotting functions as pu
import plotting_utils as pu

## Load adata

In [ ]:
adata = sc.read_h5ad(os.path.join(dist_out_dir,'adata_distance_zones_structure.h5ad'))

In [ ]:
adata.obs.columns

## Load palettes

In [ ]:
with open(os.path.join(inputs_dir, '20250519_celltype_palette_mapped.pkl'), 'rb') as f:
    celltype_palette_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, '20250520_celltype_palette_coarse_mapped.pkl'), 'rb') as f:
    celltype_palette_coarse_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, '20250520_celltype_palette_coarser_mapped.pkl'), 'rb') as f:
    celltype_palette_coarser_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, '20250521_zone_consol_palette_mapped.pkl'), 'rb') as f:
    zone_consol_palette_mapped = pickle.load(f)

In [ ]:
import matplotlib.colors as clr
import colorcet

zissou = [
    "#3A9AB2",
    "#6FB2C1",
    "#91BAB6",
    "#A5C2A3",
    "#BDC881",
    "#DCCB4E",
    "#E3B710",
    "#E79805",
    "#EC7A05",
    "#EF5703",
    "#F11B00",
]

colormap = clr.LinearSegmentedColormap.from_list("Zissou", zissou)
colormap_r = clr.LinearSegmentedColormap.from_list("Zissou", zissou[::-1])

# Distances
- Try two approaches:
- (1) distance to tls, bronchi, or vessel zone (not structure). Cells in zone = 0
- (2) distance to tls or bronchi structure

In [ ]:
def cell_dist_to_zone(adata, cell_type_col, zone_col, zone_label, nearest_n, dist_col):
    
    # get coordinates of other cell type (the one we want to calculate the distance to)
    adata_other_type = adata[adata.obs[zone_col] == zone_label, :]
    zone_label_coords = adata_other_type.obs[['x_centroid', 'y_centroid']].values
    print('adata_other_type shape ', adata_other_type.shape)

    # first set all cells in adata with zone_col == zone_label to 0
    adata.obs.loc[adata.obs[zone_col] == zone_label, dist_col] = 0
    # now only calculate distances for other zones
    adata_other_zones = adata[adata.obs[zone_col] != zone_label, :]
    
    # Iterate over each unique cell type
    for cell_type in adata_other_zones.obs[cell_type_col].unique():

        # Filter AnnData object for the current cell type
        adata_filtered = adata_other_zones[adata_other_zones.obs[cell_type_col] == cell_type, :]
        adata_filtered_cell_type_indices = adata_filtered.obs.index
        # print('adata_cell_type shape ', adata_filtered.shape)
    
        # Get the coordinates of the cells of the current type
        cell_type_coords = adata_filtered.obs[['x_centroid', 'y_centroid']].values
        
        # Calculate the pairwise distances between the current cell type and other cell type
        pairwise_distances = cdist(cell_type_coords, zone_label_coords, metric='euclidean')
    
        # Sort distances and select nearest n
        nearest_distances = np.sort(pairwise_distances, axis=1)[:, :nearest_n]
        
        # Calculate the average distance to the nearest n cells
        avg_distance = np.mean(nearest_distances, axis=1)

        # Store distances in adata.obs 
        
        # Assigning values to the new column in the original DataFrame, using cell type filtered adata indices
        adata.obs.loc[adata_filtered_cell_type_indices, dist_col] = avg_distance

    return adata

In [ ]:
import numpy as np
from shapely.geometry import Point, Polygon, MultiPolygon
from tqdm import  tqdm

def calculate_distances_to_regions(adata, dist_out_dir):
    """
    Calculate distances from each cell to the closest TLS and bronchi regions.
    Also determine if cells are inside these regions.
    
    Parameters
    ----------
    adata : AnnData
        The AnnData object containing spatial coordinates of cells
    dist_out_dir : str
        Directory where polygon files are stored
    
    Returns
    -------
    adata : AnnData
        Updated AnnData object with distance and boolean columns
    """
    # Create new columns in adata.obs for storing distances and boolean flags
    adata.obs['distance_to_tls'] = np.nan
    adata.obs['distance_to_bronchi'] = np.nan
    adata.obs['inside_tls'] = False
    adata.obs['inside_bronchi'] = False
    
    # Process each sample separately
    for sample_label in adata.obs['sample_label'].cat.categories:
        print(f"Processing sample: {sample_label}")
        
        # Get cells for current sample
        sample_mask = adata.obs['sample_label'] == sample_label
        
        # Load TLS polygons for this sample
        tls_poly_path = os.path.join(dist_out_dir, f'tls_poly_{sample_label}.pkl')
        tls_poly = joblib.load(tls_poly_path)
        
        # Load bronchi polygons for this sample
        bronchi_poly_path = os.path.join(dist_out_dir, f'bronchi_poly_{sample_label}.pkl')
        bronchi_poly = joblib.load(bronchi_poly_path)
        
        # Get spatial coordinates for cells in this sample
        coords = adata[sample_mask].obsm['spatial']
        
        # Calculate distances and check if cells are inside regions
        distances_to_tls = []
        distances_to_bronchi = []
        inside_tls_flags = []
        inside_bronchi_flags = []
        
        for i, (x, y) in tqdm(enumerate(coords), total=len(coords), desc=f"Processing {sample_label}", ncols=100):
            # Create point for current cell
            point = Point(x, y)
            
            # Check TLS regions
            min_dist_tls = float('inf')
            is_inside_tls = False
            
            # Handle potential MultiPolygon case
            if isinstance(tls_poly, MultiPolygon):
                tls_polygons = list(tls_poly.geoms)
            else:
                tls_polygons = [tls_poly]
                
            for poly in tls_polygons:
                # Check if point is inside polygon
                if poly.contains(point):
                    min_dist_tls = 0
                    is_inside_tls = True
                    break
                # Calculate distance to polygon if not inside
                dist = point.distance(poly)
                if dist < min_dist_tls:
                    min_dist_tls = dist
            
            # Check bronchi regions
            min_dist_bronchi = float('inf')
            is_inside_bronchi = False
            
            # Handle potential MultiPolygon case
            if isinstance(bronchi_poly, MultiPolygon):
                bronchi_polygons = list(bronchi_poly.geoms)
            else:
                bronchi_polygons = [bronchi_poly]
                
            for poly in bronchi_polygons:
                # Check if point is inside polygon
                if poly.contains(point):
                    min_dist_bronchi = 0
                    is_inside_bronchi = True
                    break
                # Calculate distance to polygon if not inside
                dist = point.distance(poly)
                if dist < min_dist_bronchi:
                    min_dist_bronchi = dist
            
            distances_to_tls.append(min_dist_tls)
            distances_to_bronchi.append(min_dist_bronchi)
            inside_tls_flags.append(is_inside_tls)
            inside_bronchi_flags.append(is_inside_bronchi)
        
        # Update AnnData object with calculated values for this sample
        adata.obs.loc[sample_mask, 'distance_to_tls'] = distances_to_tls
        adata.obs.loc[sample_mask, 'distance_to_bronchi'] = distances_to_bronchi
        adata.obs.loc[sample_mask, 'inside_tls'] = inside_tls_flags
        adata.obs.loc[sample_mask, 'inside_bronchi'] = inside_bronchi_flags
        
    return adata

In [ ]:
for zone_label in ['bronchi', 'vessels', 'TLS']:
    cell_type_col = 'label_fine'
    zone_col = 'zone_consol'
    dist_col = f'avg_distance_to_{zone_label}_zone'
    nearest_n = 10 
    
    start_time = time.time()
    
    adata.obs[dist_col] = np
    
    for sample_label in adata.obs['sample_label'].cat.categories:
        
        print(sample_label)
    
        adata_sample = adata[adata.obs['sample_label']==sample_label, :]
        
        # run distance calculation for each sample 
        adata_sample = cell_dist_to_zone(adata=adata_sample, 
                                                   cell_type_col=cell_type_col, 
                                                   zone_col=zone_col, 
                                                   zone_label=zone_label, 
                                                   nearest_n=nearest_n, 
                                                   dist_col=dist_col)
        
        # get distance df for each structure and add it back to the main adata object
        dist_df = adata_sample.obs[[dist_col]] 
        adata.obs.loc[dist_df.index, dist_col] = dist_df[dist_col]
    
    adata.obs[dist_col] = pd.to_numeric(adata.obs[dist_col])
    end_time = time.time()
    elapsed_time = (end_time - start_time) / 60
    print(elapsed_time, ' min')

calculate_distances_to_regions(adata, dist_out_dir)

# adata.write_h5ad(os.path.join(dist_out_dir,'adata_distance_zones_structure.h5ad'), compression='gzip')

# Plot distances

In [ ]:
adata = sc.read_h5ad(os.path.join(dist_out_dir,'adata_distance_zones_structure.h5ad'))

In [ ]:
# matplotlib settings
plt.rcdefaults()
mpl.rcParams['pdf.fonttype'] = 42

In [ ]:
import scvelo as scv
def scvelo_heatmap(
    adata: sc.AnnData,
    sample_label: list[str],
    sortby: str,
    key_name: str,
    key_value: str = None,
    highlight: list[str] = None,
    n_bins: int = 5,
    col_color = None, 
    x_clip = None,
    expression_threshold = None, 
    figsize: tuple = None 
):
    """
    Create a heatmap to visualize gene expression trends in single-cell RNA-seq data,
    with options for subsetting, sorting, and highlighting genes.

    Parameters:
    - adata (sc.AnnData): Annotated data object containing single-cell RNA-seq data.
    - sample_label (List[str]): List of batch identifiers to subset the data.
    - key_name (str): String representing the key in `adata.obs` to use for subsetting cells.
    - key_value (str): String representing the value of `key_name` to subset to.
    - sortby (str): Variable to sort the heatmap by (e.g., "crypt_villi_axis").
    - highlight (List[str]): List of labels to highlight on the heatmap.
    - n_bins (int, optional): Integer specifying the number of bins to use for convolution (default: 5).

    Returns:
    - s (seaborn.matrix.ClusterGrid): Matplotlib figure object representing the heatmap.

    This function subsets the input data based on specified sample_labels and key-value pairs,
    filters genes expressed in a minimum percentage of cells, and creates a heatmap
    to visualize gene expression trends along a specified variable. The function also allows
    highlighting specific labels on the y-axis.
    """
    print("Creating Heatmap for sample labels ", " + ".join(sample_label))

    # Subset sample_label
    adata = adata[adata.obs["sample_label"].isin(sample_label)]
    if key_value:
        print(f"Subset to '{key_name}'=='{key_value}'")
        # Subset to key
        adata = adata[adata.obs[key_name] == key_value]
    else:
        print("using all cells")
        
    # Filter to include only genes that are expressed in 5% of the cells
    if expression_threshold:
        print('removing genes by expression threshold: ', expression_threshold, ' %')
        adata=filter_adata_expressed_in_n_cells(adata,percent=expression_threshold)
    else:
        adata = adata.copy()
    print('adata shape: ', adata.shape)

    # Clip x axis
    if x_clip:
        print('clipping x to ', x_clip)
        adata = adata[adata.obs[sortby] <= x_clip, :]
        # adata.obs.loc[adata.obs[sortby] > x_clip, sortby] = x_clip
        
    n_convolve = len(adata) // n_bins
    print(f"Setting `n_convolve` to {n_convolve} ({n_bins} bins, {len(adata)} cells) ")
    # Plot
    s = scv.pl.heatmap(
        adata,
        var_names=adata.var_names,
        sortby=sortby,
        n_convolve=n_convolve,
        show=False,
        yticklabels=True,
        rasterized=True,
        # color_map=colormap,
        color_map='viridis',
        col_color = col_color,
        figsize=figsize
        
    )
    ax = s.ax_heatmap

    # Loop through the x-axis tick labels and show/hide based on the 'highlight' list
    if highlight:
        for i, label in enumerate(ax.get_yticklabels()):
            if label.get_text() not in highlight:
                label.set_visible(False)
                ax.get_yticklines()[2 * i + 1].set_visible(False)
            ax.get_yticklines()[2 * i].set_visible(False)
    else:
        # Make sure all labels and ticks are visible
        for label in ax.get_yticklabels():
            label.set_visible(True)
    
        for tickline in ax.get_yticklines():
            tickline.set_visible(False)  # optional: hide all tick lines
        

    ax.set_xlabel("")
    ax.set_title("")
    return s

## Separate by sample timepoint 

In [ ]:
adata_d3 = adata[adata.obs['sample_label']=='HDM_day3', :]
adata_d30 = adata[adata.obs['sample_label']=='HDM_day30', :]

In [ ]:
sc.pl.embedding(adata_d3, basis = 'spatial', color = 'avg_distance_to_bronchi_zone',
               vmax = 500)
sc.pl.embedding(adata_d3, basis = 'spatial', color = 'distance_to_bronchi',
               vmax = 500)

In [ ]:
sc.pl.embedding(adata_d3, basis = 'spatial', color = 'distance_to_tls', vmax = 500)
sc.pl.embedding(adata_d3, basis = 'spatial', color = 'avg_distance_to_TLS_zone', vmax = 500)
sc.pl.embedding(adata_d3, basis = 'spatial', color = 'tls_index')
sc.pl.embedding(adata_d3, basis = 'spatial', color = 'zone_consol')

In [ ]:
sc.pl.embedding(adata_d3, basis = 'spatial', color = 'avg_distance_to_vessels_zone', vmax = 300)


## Plot CD4 DE genes along distance axes (in CD4 act only)

## First, make a quick colorbar

In [ ]:
import matplotlib.cm as cm
# Define the colormap
cmap = cm.viridis  # or e.g., cm.viridis_r, cm.Reds, etc.

# Create a dummy scalar mappable for the colorbar
norm = mcolors.Normalize(vmin=0, vmax=1)
sm = cm.ScalarMappable(norm=norm, cmap=cmap)

# Create a new figure and axis
fig, ax = plt.subplots(figsize=(1, 4))  # adjust size as needed

# Add the colorbar to the axis
cb = fig.colorbar(sm, cax=ax)

# Remove ticks and labels
cb.ax.set_yticks([])
cb.ax.set_yticklabels([])

# Remove outline
cb.outline.set_visible(False)

# Save the figure
fig.savefig(os.path.join(plot_out_dir, "colormap_viridis_clean.png"), dpi=300, bbox_inches='tight', transparent=True)
plt.close(fig)

#### Filter genes by those robustly expressed in CITE dataset and in Xenium datast (5% threshold in at least one subset):

In [ ]:
# CITE genes expressed in 5% of at least one subset
gene_df = pd.read_csv('/home/workspace/spatial_mouse_lung_outputs/distance_analysis/ouputs_mouselung_xenseg_HDM/R/cite_expressed_genes_by_annotation.csv')
genes_cite_filt = gene_df['x'].tolist()

# Xenium genes expressed in 5% of at least one subset
genes_xen_filt = []
adata_cd4 = adata.copy()
for subset in adata_cd4.obs['label_fine'].unique().tolist():
    adata_cd4_subset = adata_cd4[adata_cd4.obs['label_fine'] == subset, :]
    adata_cd4_subset = filter_adata_expressed_in_n_cells(adata_cd4_subset, percent=0.05)
    genes_xen_filt.extend(adata_cd4_subset.var.index.tolist())
genes_xen_filt = list(set(genes_xen_filt))

# take the intersection and use this list for heatmaps 
genes_cite_xen_filt = list(set(genes_cite_filt) & set(genes_xen_filt))

In [ ]:
genes_highlight_tls = ['Il21', 'Ccr7', 'Slamf6', 'Tcf7', 'Id3', 'Tbx21', 'Cxcr6', 'Il1rl1', 'Gata3', 'Il13', 'Calca']
genes_highlight_bronchi = ['Il1rl1', 'Il2ra', 'Foxp3', 'Gata3', 'Klrg1', 'Nmur1', 'Prdm1', 'Cxcr6', 'Id3', 'Slamf6', 'Tcf7', 'Tbx21', 'Ccr7']

In [ ]:
for sample_label in adata.obs['sample_label'].cat.categories:
    print(sample_label)
    adata_plot = adata[adata.obs['sample_label']==sample_label, :]
    s = scvelo_heatmap(
        adata_plot[:,genes_cite_xen_filt],
        sample_label=[sample_label],
        sortby = 'distance_to_tls',
        key_name = 'label_medium',
        key_value = 'CD4 act',
        highlight = genes_highlight_tls,
        x_clip = 450, 
        expression_threshold= None, # already using a filtered gene list 
        figsize=(5,6)
    )
    s
    plt.savefig(os.path.join(plot_out_dir, f"{sample_label}_scVelo_tls-structure_CD4.pdf"), dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
for sample_label in adata.obs['sample_label'].cat.categories:
    print(sample_label)
    adata_plot = adata[adata.obs['sample_label']==sample_label, :]
    s = scvelo_heatmap(
        adata_plot[~adata_plot.obs['zone_consol'].isin(['TLS']),genes_cite_xen_filt],
        sample_label=[sample_label],
        sortby = 'avg_distance_to_bronchi_zone',
        key_name = 'label_medium',
        key_value = 'CD4 act',
        highlight = genes_highlight_bronchi,
        x_clip = 450, 
        expression_threshold= None, # already using a filtered gene list 
        figsize=(5,6)
    )
    s
    plt.savefig(os.path.join(plot_out_dir, f"{sample_label}_scVelo_bronchi-zone_CD4.pdf"), dpi=300, bbox_inches='tight', transparent=True)

### Generate heatmaps showing all gene labels, just for day 3

In [ ]:
sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
s = scvelo_heatmap(
    adata_plot[:,genes_cite_xen_filt],
    sample_label=[sample_label],
    sortby = 'distance_to_tls',
    key_name = 'label_medium',
    key_value = 'CD4 act',
    highlight = None,
    x_clip = 500, 
    expression_threshold= None, # already using a filtered gene list 
    figsize=(5,25)
)
plt.savefig(os.path.join(plot_out_dir, f"{sample_label}_scVelo_tls-structure_CD4_showallgenes.pdf"), dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
s = scvelo_heatmap(
    adata_plot[~adata_plot.obs['zone_consol'].isin(['TLS']),genes_cite_xen_filt],
    sample_label=[sample_label],
    sortby = 'avg_distance_to_bronchi_zone',
    key_name = 'label_medium',
    key_value = 'CD4 act',
    highlight = None,
    x_clip = 500, 
    expression_threshold= None, # already using a filtered gene list 
    figsize=(5,25)
)
plt.savefig(os.path.join(plot_out_dir, f"{sample_label}_scVelo_bronchi-zone_CD4_showallgenes.pdf"), dpi=300, bbox_inches='tight', transparent=True)

## Plot cytokine ligand DE genes along distance axes (in all cells)

In [ ]:
# try cellphoneDB ligands instead of kegg cytokine ligands

df_lr = pd.read_csv(os.path.join(inputs_dir, 'interaction_input_CellChatDB.csv'))
ligands = df_lr['ligand'].tolist()
ligands_panel = list(set(ligands) & set(adata_d3.var.index))
len(ligands_panel) 

In [ ]:
# df_kegg_cytokines = pd.read_csv("/home/workspace/spatial_tls_manuscript/inputs/kegg_cytokines.csv")
# df_kegg_cytokines = df_kegg_cytokines[df_kegg_cytokines["type"] == "ligand"]

# all_cyto_genes = df_kegg_cytokines['gene'].tolist()

# cyto_genes_panel = list(set(all_cyto_genes) & set(adata_d3.var.index))
# len(cyto_genes_panel)

### All cytokine ligands in panel 

In [ ]:
s = scvelo_heatmap(
    # adata_d3[:,cyto_genes_panel],
    adata_d3[:,ligands_panel],
    sample_label=['HDM_day3'],
    sortby = 'distance_to_tls',
    key_name = 'label_medium',
    key_value = None,
    highlight = None,
    x_clip = 450,
    expression_threshold= 0.05,
    figsize=(5,4)
)
s
# plt.savefig(os.path.join(plot_out_dir, "HDM_day3_scVelo_tls-structure_allcells_cyto.pdf"), dpi=300, bbox_inches='tight', transparent=True)
plt.savefig(os.path.join(plot_out_dir, "HDM_day3_scVelo_tls-structure_allcells_cellphoneDBligands.pdf"), dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
s = scvelo_heatmap(
    # adata_d3[:,cyto_genes_panel],
    adata_d3[:,ligands_panel],
    sample_label=['HDM_day3'],
    sortby = 'avg_distance_to_bronchi_zone',
    key_name = 'label_medium',
    key_value = None,
    highlight = None,
    x_clip = 450,
    expression_threshold= 0.05,
    figsize=(5,4)
)
s
plt.savefig(os.path.join(plot_out_dir, "HDM_day3_scVelo_bronchi_allcells_cellphoneDBligands.pdf"), dpi=300, bbox_inches='tight', transparent=True)

## Define categories based on TLS and bronchi proximity

### Gate cell subsets by spatial region (2D distance density)

Plot the 2D distance density of a chosen `label_fine` (or `label_medium`) subset, overlay rectangular **gates**, iterate on the gate bounds, then apply the final gates to write a `spatial_region` column to `adata.obs`.

- `plot_celltype_density_2d(...)` — KDE density of one cell subset over (bronchi-distance, TLS-distance), with gates drawn on top.
- `assign_spatial_region(...)` — applies the same gates to label every cell with its region.

In [ ]:
import matplotlib.patches as mpatches
from functools import partial


# --- Axis transforms ----------------------------------------------------------
# Biexponential-style transforms spread out the region near a zone boundary so
# that cells AT a zone (distance == 0) are visually separable from cells just
# outside it (e.g. 0 < distance_to_tls < 100), while still compressing the long
# tail of large distances.
#
# Transforms are applied ONLY for plotting. The `gates` bounds and
# `assign_spatial_region` stay in RAW distance units: because each transform is
# monotonic and applied independently per axis, axis-aligned rectangular gates
# select the exact same cells whether evaluated in raw or transformed space, so
# what you draw is still what you assign.

def arcsinh_transform(x, linthresh=30.0):
    """Biexponential ('logicle'-style) transform: ~linear for x << linthresh,
    ~logarithmic for x >> linthresh. Smaller `linthresh` => more expansion of
    the near-zero region. Handles 0 exactly (arcsinh(0) == 0). Good default for
    separating distance==0 from small positive distances on the 0-450 scale."""
    return np.arcsinh(np.asarray(x, dtype=float) / linthresh)


def biexp_transform(x, a=0.1, b=0.1, c=0.5, d=2.5, f=4, w=1):
    """Custom biexponential transform (same form as Spatial-TRM-paper Fig 4).
    Monotonic for positive a, b, c, d. Parameters are data-scale specific and
    generally need tuning for the 0-450 distance range used here -- prefer
    `arcsinh_transform` unless you specifically need this shape."""
    x = np.asarray(x, dtype=float)
    return a * np.exp(b * (x - w)) - c * np.exp(-d * (x - w)) + f


def _as_transform(t):
    """Normalize a transform argument: None -> identity callable."""
    if t is None:
        return lambda v: np.asarray(v, dtype=float)
    return t


def _draw_gates(ax, gates, x_clip, y_clip, x_transform=None, y_transform=None,
                show_labels=True):
    """Overlay named rectangular gates on a 2D distance plot.

    Each gate is {'x_min','x_max','y_min','y_max'} (any subset of keys), given
    in RAW distance units. Missing/None bounds default to the plot edges
    (0 .. x_clip / y_clip). Bounds are mapped through `x_transform` /
    `y_transform` so the drawn rectangle lines up with the transformed data."""
    xt, yt = _as_transform(x_transform), _as_transform(y_transform)
    for name, b in gates.items():
        x0 = b.get('x_min') if b.get('x_min') is not None else 0
        x1 = b.get('x_max') if b.get('x_max') is not None else x_clip
        y0 = b.get('y_min') if b.get('y_min') is not None else 0
        y1 = b.get('y_max') if b.get('y_max') is not None else y_clip
        tx0, tx1 = float(xt(x0)), float(xt(x1))
        ty0, ty1 = float(yt(y0)), float(yt(y1))
        ax.add_patch(mpatches.Rectangle(
            (tx0, ty0), tx1 - tx0, ty1 - ty0,
            fill=False, edgecolor='black', linestyle='--', linewidth=2,
        ))
        if show_labels:
            ax.text((tx0 + tx1) / 2, (ty0 + ty1) / 2, name,
                    ha='center', va='center', fontsize=8, color='black')


def plot_celltype_density_2d(
    adata: sc.AnnData,
    sample_label: list[str],
    celltype: str,
    x_axis: str = 'avg_distance_to_bronchi_zone',
    y_axis: str = 'distance_to_tls',
    celltype_col: str = 'label_fine',
    gates: dict = None,
    x_clip: float = 450,
    y_clip: float = 450,
    x_transform=None,
    y_transform=None,
    x_ticks: list = None,
    y_ticks: list = None,
    figsize: tuple = (3, 3),
    point_size: float = 2,
    show_gate_labels: bool = True,
):
    """
    2D distance plot of the *density* of a single cell subset.

    Like `distance_xy_kde`, but instead of weighting by gene expression this
    shows where cells of `celltype` (from `adata.obs[celltype_col]`)
    concentrate in the (x_axis, y_axis) distance space. Each subset cell is a
    point colored by its local 2D Gaussian-KDE density, with KDE contours.

    Optionally overlays rectangular `gates` (see `_draw_gates`) so different
    region definitions can be eyeballed before applying `assign_spatial_region`.

    Axis transforms
    ---------------
    Pass `x_transform` and/or `y_transform` (a callable mapping raw distance ->
    plotted position, e.g. `partial(arcsinh_transform, linthresh=30)`) to spread
    out the near-zero region so distance==0 separates from small positive
    distances. Transforms are applied to the data, the KDE, and the gate
    rectangles. Gates and `assign_spatial_region` remain in raw units. `None`
    means no transform (identity / linear axis).

    Use `x_ticks` / `y_ticks` to label the (possibly nonlinear) axes at chosen
    RAW distance values, e.g. `y_ticks=[0, 25, 50, 100, 450]`. If omitted, the
    axis is left unlabeled with ticks at the clip extremes (previous behavior).

    Parameters
    ----------
    adata : AnnData
    sample_label : list[str]
        sample_label values to include.
    celltype : str
        Value in adata.obs[celltype_col] whose density is plotted.
    celltype_col : str
        obs column holding the cell-type labels (default 'label_fine').
    x_axis, y_axis : str
        Distance columns for the x and y axes.
    gates : dict, optional
        region name -> {'x_min','x_max','y_min','y_max'} bounds (raw units) to overlay.
    x_clip, y_clip : float
        Axis limits in RAW units; cells beyond these are dropped from the plot.
    x_transform, y_transform : callable, optional
        Monotonic raw-distance -> plotted-position maps. None = linear.
    x_ticks, y_ticks : list, optional
        RAW distance values at which to place labeled ticks.
    point_size : float
        Scatter point size.

    Returns
    -------
    plt : module
        The pyplot module (call .show() or .savefig()).
    """
    fig = plt.figure(figsize=figsize, dpi=200)

    xt, yt = _as_transform(x_transform), _as_transform(y_transform)

    # Subset by sample_label
    adata = adata[adata.obs['sample_label'].isin(sample_label)]

    # Clip to the plotting window in RAW units (matches distance_xy_kde behavior)
    if x_clip:
        adata = adata[adata.obs[x_axis] <= x_clip, :]
    if y_clip:
        adata = adata[adata.obs[y_axis] <= y_clip, :]

    # Subset to the cell type of interest
    print(f"Subset to '{celltype_col}' == '{celltype}'")
    sub_adata = adata[adata.obs[celltype_col] == celltype].copy()
    print('cells in subset: ', sub_adata.n_obs)

    # Apply axis transforms (identity if None)
    x = xt(sub_adata.obs[x_axis].values)
    y = yt(sub_adata.obs[y_axis].values)

    ax = fig.add_subplot(1, 1, 1)

    # color each subset cell by its local 2D density (in transformed space, so
    # the coloring matches the plotted point positions)
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)
    ax.scatter(x, y, c=z, s=point_size, marker='.', cmap='viridis')

    # density contours of the subset
    sns.kdeplot(x=x, y=y, ax=ax, color='#444444', linewidths=1)

    # overlay gates (transformed to match the data)
    if gates:
        _draw_gates(ax, gates, x_clip, y_clip,
                    x_transform=x_transform, y_transform=y_transform,
                    show_labels=show_gate_labels)

    # axis limits in transformed space with a small relative pad
    tx_lo, tx_hi = float(xt(0)), float(xt(x_clip))
    ty_lo, ty_hi = float(yt(0)), float(yt(y_clip))
    xpad = 0.02 * (tx_hi - tx_lo)
    ypad = 0.02 * (ty_hi - ty_lo)
    ax.set_xlim(tx_lo - xpad, tx_hi + xpad)
    ax.set_ylim(ty_lo - ypad, ty_hi + ypad)

    # ticks: label chosen RAW distances at their transformed positions, else
    # keep the previous unlabeled clip-extent ticks
    if x_ticks is not None:
        ax.set_xticks([float(xt(t)) for t in x_ticks])
        ax.set_xticklabels([str(t) for t in x_ticks])
    else:
        ax.set_xticks([tx_lo, tx_hi])
        ax.set_xticklabels([])
    if y_ticks is not None:
        ax.set_yticks([float(yt(t)) for t in y_ticks])
        ax.set_yticklabels([str(t) for t in y_ticks])
    else:
        ax.set_yticks([ty_lo, ty_hi])
        ax.set_yticklabels([])

    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.grid(False)
    plt.title(celltype, fontsize=12)
    fig.tight_layout()
    return plt

In [ ]:
def assign_spatial_region(
    adata: sc.AnnData,
    gates: dict,
    x_axis: str = 'avg_distance_to_bronchi_zone',
    y_axis: str = 'distance_to_tls',
    region_col: str = 'spatial_region',
    default: str = 'unassigned',
):
    """
    Label every cell with the spatial region it falls into, based on `gates`.

    Uses the same gate definition as `plot_celltype_density_2d` so what you
    draw is what you assign. Applied to ALL cells in `adata` (no clipping),
    writing the result to `adata.obs[region_col]` as an ordered Categorical.

    A cell is in a gate if its (x_axis, y_axis) coordinates satisfy every
    bound present in that gate: x_min <= x < x_max and y_min <= y < y_max
    (missing bounds are treated as open). Gates are applied in dict order
    and the LAST matching gate wins, so list overlapping gates from most
    general to most specific. Cells matching no gate get `default`.

    Parameters
    ----------
    adata : AnnData
    gates : dict
        region name -> {'x_min','x_max','y_min','y_max'} (any subset; None = open).
    x_axis, y_axis : str
        Columns in adata.obs holding the distance coordinates.
    region_col : str
        Output column written to adata.obs.
    default : str
        Label for cells not captured by any gate.

    Returns
    -------
    adata : AnnData
        The same object, with adata.obs[region_col] added.
    """
    x = adata.obs[x_axis].values
    y = adata.obs[y_axis].values
    region = np.full(adata.n_obs, default, dtype=object)

    for name, b in gates.items():
        mask = np.ones(adata.n_obs, dtype=bool)
        if b.get('x_min') is not None:
            mask &= x >= b['x_min']
        if b.get('x_max') is not None:
            mask &= x < b['x_max']
        if b.get('y_min') is not None:
            mask &= y >= b['y_min']
        if b.get('y_max') is not None:
            mask &= y < b['y_max']
        region[mask] = name

    categories = list(gates.keys())
    if (region == default).any():
        categories = categories + [default]
    adata.obs[region_col] = pd.Categorical(region, categories=categories, ordered=True)
    return adata

In [ ]:
# --- Define gates and visualize them over a cell subset's density ---
# Each gate is a named region with optional x_min/x_max/y_min/y_max bounds, in
# RAW distance units. Omit (or set None) a bound to leave that edge open. Edit
# freely to test different region definitions, then re-run the assignment cell.
# x_axis = avg_distance_to_bronchi_zone, y_axis = distance_to_tls
gates = {
    'near-TLS (T)':     {'y_max': 100},                    # close to TLS (any bronchi dist)
    'near-bronchi (B)': {'x_max': 150, 'y_min': 100},      # close to bronchi, far from TLS
    'parenchyma (P)':   {'x_min': 150, 'y_min': 100},      # far from both
}

# --- Optional biexponential axis transforms ---
# Spread out the near-zero region so cells AT a zone (distance == 0) separate
# visually from cells just outside it (e.g. 0 < distance_to_tls < 100). Set a
# transform to None for a plain linear axis. `linthresh` sets how wide the
# ~linear near-zero band is (smaller => more expansion of small distances).
# Gates and assign_spatial_region stay in raw units, so the assignment below is
# unaffected by these display transforms.
y_transform = partial(arcsinh_transform, linthresh=30)   # TLS distance: separate 0 from <100
x_transform = None                                       # bronchi distance: linear (try arcsinh too)
# To biexp BOTH axes, e.g.: x_transform = partial(arcsinh_transform, linthresh=50)

sample_label = 'HDM_day3'
adata_plot = adata[adata.obs['sample_label'] == sample_label, :]

p = plot_celltype_density_2d(
    adata_plot,
    sample_label=[sample_label],
    celltype='CD4 act',
    celltype_col='label_medium',   # use 'label_fine' for finer subsets
    x_axis='avg_distance_to_bronchi_zone',
    y_axis='distance_to_tls',
    gates=gates,
    x_clip=450,
    y_clip=450,
    x_transform=x_transform,
    y_transform=y_transform,
    x_ticks=[0, 150, 450],            # raw-distance values to label on each axis
    y_ticks=[0, 25, 50, 100, 200, 450],
    point_size=24,
)
# plt.savefig(os.path.join(plot_out_dir, f"2D_density_CD4act_gates_{sample_label}.pdf"), dpi=300, bbox_inches='tight', transparent=True)
p.show()

In [ ]:
# Apply the gates to label every cell with the region it falls in.
# This writes adata.obs[region_col] for ALL cells (no clipping), so each
# cell gets a region using the same gate bounds drawn above.
adata = assign_spatial_region(
    adata,
    gates=gates,
    x_axis='avg_distance_to_bronchi_zone',
    y_axis='distance_to_tls',
    region_col='spatial_region',
)

adata.obs['spatial_region'].value_counts(dropna=False)

## 2D distance plots

In [ ]:
from scipy.spatial import KDTree
from scipy.stats import gaussian_kde

def scatter_with_gaussian_kde(ax, x, y, weights=None, s=2, **kwargs):
    xy = np.vstack([x, y])
    try:
        z = gaussian_kde(xy, weights=weights)(xy)
    except:
        z = gaussian_kde(xy)(xy)

    kwargs.setdefault('marker', '.')
    ax.scatter(x, y, c=z, s=s, **kwargs)

def distance_xy_kde(
    adata: sc.AnnData,
    sample_label: list[str],
    x_axis: str,
    y_axis: str,
    key_name: str,
    key_value: str = None,
    gene_name: str = None,
    signature_col: str = None, 
    palette=None, 
    x_clip=None,
    y_clip=None,
    figsize: tuple = None,
    vline: float = None, 
    hline: float = None,
    point_size = 2,
):

    fig = plt.figure(figsize=(3, 3), dpi=200)
    
    # Subset by sample_label
    adata = adata[adata.obs["sample_label"].isin(sample_label)]
    
    if key_value:
        print(f"Subset to '{key_name}' == '{key_value}'")
        adata = adata[adata.obs[key_name] == key_value]
    else:
        print("Using all cells")

    # Clip x and y axes
    if x_clip:
        print('Clipping x to', x_clip)
        adata = adata[adata.obs[x_axis] <= x_clip, :]
    if y_clip:
        print('Clipping y to', y_clip)
        adata = adata[adata.obs[y_axis] <= y_clip, :]

    sub_adata = adata.copy()

    if signature_col:
        print('plotting signature')
        gene_expr = sub_adata.obs[signature_col]
        title = signature_col
    else:
        print('plotting gene expr')
        gene_expr = sub_adata[:, gene_name].X.toarray().flatten() if hasattr(sub_adata[:, gene_name].X, "toarray") else sub_adata[:, gene_name].X.flatten()
        gene_expr = np.nan_to_num(gene_expr)
        title = gene_name
    
    ax = fig.add_subplot(1, 1, 1)
    
     
    # KDE scatter plot
    scatter_with_gaussian_kde(
        ax=ax,
        x=sub_adata.obs[x_axis],
        y=sub_adata.obs[y_axis],
        s=point_size,
        weights=(gene_expr - np.min(gene_expr)) ** 2,
        cmap="viridis",
    )

    # KDE weighted by expression
    sns.kdeplot(
        data=sub_adata.obs,
        x=x_axis,
        y=y_axis,
        ax=ax,
        color="#444444",
        linewidths=1,
        weights=(gene_expr - np.min(gene_expr)) ** 2,
        cmap="viridis",
    )

    # Add vline and hline if specified
    if vline is not None:
        # ax.axvline(x=vline, color='black', linestyle='--', linewidth=2)
        ax.plot([vline, vline], [hline, y_clip], color='black', linestyle='--', linewidth=2)
    if hline is not None:
        ax.axhline(y=hline, color='black', linestyle='--', linewidth=2)


    ax.set_xlim(-10,x_clip+10)
    ax.set_ylim(-10,y_clip+10)
    
    ax.set_xticks([0, x_clip])
    ax.set_yticks([0, y_clip])
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    # ax.set_xticklabels(['0', str(x_clip)], fontsize=16)
    # ax.set_yticklabels(['0', str(y_clip)], fontsize=16)    
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.grid(False)

    fig.tight_layout()
    # plt.title(title, fontsize=20)
    plt.title('')
    
    return(plt)


In [ ]:
sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
for gene_name in ['Pdcd1' ]:
    p = distance_xy_kde(adata_plot,
                    sample_label=[sample_label],
                    x_axis = 'avg_distance_to_bronchi_zone',
                        # y_axis = 'avg_distance_to_TLS_zone',
                        y_axis = 'distance_to_tls',
                        gene_name = gene_name,
                    key_name = 'label_medium',
                    key_value = 'CD4 act',
                   x_clip = 450,
                   y_clip= 450,
                        vline = 150, hline = 100
                       )
    # plt.savefig(os.path.join(plot_out_dir, f"2D_{gene_name}.pdf"), dpi=300, bbox_inches='tight', transparent=True)


In [ ]:
sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
adata_plot_cd4 = adata_plot[adata_plot.obs['label_medium']=='CD4 act', :]
adata_plot_cd4.shape

In [ ]:
sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
for gene_name in ['Gata3', 
                  'Il1rl1', 'Klrg1',
                  'Tcf7', 'Slamf6', 'Ccr7',
                 ]:
    p = distance_xy_kde(adata_plot,
        sample_label=[sample_label],
        x_axis = 'avg_distance_to_bronchi_zone',
        y_axis = 'distance_to_tls',
        gene_name = gene_name,
        key_name = 'label_medium',
        key_value = 'CD4 act',
        x_clip = 450,
        y_clip= 450,
        vline = 150, hline = 100,
        point_size = 24
                       )
    # plt.savefig(os.path.join(plot_out_dir, f"2D_{gene_name}.pdf"), dpi=300, bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(plot_out_dir, f"2D_{gene_name}.png"), dpi=300, bbox_inches='tight', transparent=True)


### Distance quadrant gene expression heatmap

In [ ]:
def quantify_quadrant_expression(adata, 
                                 x_axis = 'avg_distance_to_bronchi_zone',
                                 y_axis = 'distance_to_tls',
                                 x_clip = 450, y_clip = 450,
                                 # x_clip=None, y_clip=None,
                                 x_threshold = 150, y_threshold = 100):
    
    adata = adata.copy()

    # Clip x and y axes
    if x_clip:
        print('Clipping x to', x_clip)
        adata = adata[adata.obs[x_axis] <= x_clip, :]
    if y_clip:
        print('Clipping y to', y_clip)
        adata = adata[adata.obs[y_axis] <= y_clip, :]
        
    quadrant_order = ['near-bronchi (B)', 'parenchyma (P)', 'near-TLS (T)']
    adata.obs['spatial_quadrant']= 'none'
    adata.obs.loc[(adata.obs[x_axis] > x_threshold) & (adata.obs[y_axis] > y_threshold), 'spatial_quadrant'] = 'parenchyma (P)'
    adata.obs.loc[adata.obs[y_axis] < y_threshold, 'spatial_quadrant'] = 'near-TLS (T)'
    adata.obs.loc[(adata.obs[x_axis] < x_threshold) & (adata.obs[y_axis] > y_threshold), 'spatial_quadrant'] = 'near-bronchi (B)'
    adata.obs['spatial_quadrant'] = pd.Categorical(adata.obs['spatial_quadrant'], categories=quadrant_order, ordered=True)
    return(adata) 

In [ ]:
sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
adata_plot = adata_plot[adata_plot.obs['label_medium']=='CD4 act', :]

adata_plot = quantify_quadrant_expression(adata_plot)

print(adata_plot.shape)
sc.pl.matrixplot(adata_plot, var_names=['Gata3', 'Il1rl1', 'Klrg1', 'Tcf7', 'Slamf6', 'Ccr7'], groupby='spatial_quadrant', 
                 standard_scale='var', 
                 show=False
                )
plt.savefig(os.path.join(plot_out_dir, "spatial_quadrant_gene_matrixplot.pdf"), dpi=300, bbox_inches='tight', transparent=True)

## Plot zones and distance in space

In [ ]:
sample_label = 'HDM_day3'
adata_plot = adata_d3.copy()
adata_plot.obs['zone_consol_temp'] = adata_plot.obs['zone_consol'].copy()
adata_plot.obs.loc[adata_plot.obs['zone_consol'] != 'TLS', 'zone_consol_temp'] = np.nan
sc.pl.embedding(adata_plot, basis='spatial', color='zone_consol_temp',
                palette = zone_consol_palette_mapped,
                frameon=False, 
                size=50, 
                title = '',
                legend_loc = 'none',
                show=False)

ax = plt.gca()
tls_poly = joblib.load(os.path.join(dist_out_dir, 'tls_poly_' + sample_label + '.pkl'))
bronchi_poly = joblib.load(os.path.join(dist_out_dir, 'bronchi_poly_' + sample_label + '.pkl'))
vessels_poly = joblib.load(os.path.join(dist_out_dir, 'vessels_poly_' + sample_label + '.pkl'))
for p in tls_poly.geoms:
        ax.plot(*p.exterior.xy, c='black', linewidth=4)
   
plt.xlim([3700, 4400])
plt.ylim([4200, 4900])
plt.savefig(os.path.join(plot_out_dir, "zone_day3_tls_zoom.png"), dpi=300, bbox_inches='tight', transparent=True)

fig = sc.pl.embedding(adata_d3, basis='spatial', 
                # color='avg_distance_to_TLS_zone',
                color = 'distance_to_tls',
                cmap = colormap_r,
                vmax=450,
                frameon=False, 
                size=50, 
                title = '',
                colorbar_loc = None,
                show=False, return_fig=True)

ax = plt.gca()
tls_poly = joblib.load(os.path.join(dist_out_dir, 'tls_poly_' + sample_label + '.pkl'))
bronchi_poly = joblib.load(os.path.join(dist_out_dir, 'bronchi_poly_' + sample_label + '.pkl'))
vessels_poly = joblib.load(os.path.join(dist_out_dir, 'vessels_poly_' + sample_label + '.pkl'))
for p in tls_poly.geoms:
        ax.plot(*p.exterior.xy, c='black', linewidth=4)
        
plt.xlim([3700, 4400])
plt.ylim([4200, 4900])
plt.savefig(os.path.join(plot_out_dir, "distance_day3_tls_zoom.png"), dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
adata_plot = adata_d3.copy()
adata_plot.obs['zone_consol_temp'] = adata_plot.obs['zone_consol'].copy()
adata_plot.obs.loc[adata_plot.obs['zone_consol'] != 'bronchi', 'zone_consol_temp'] = np.nan
sc.pl.embedding(adata_plot, basis='spatial', color='zone_consol_temp',
                palette = zone_consol_palette_mapped,
                frameon=False, 
                size=50, 
                title = '',
                legend_loc = 'none',
                show=False)
    
plt.xlim([3150, 3900])
plt.ylim([2850, 3600])
plt.savefig(os.path.join(plot_out_dir, "zone_day3_bronchi_zoom.png"), dpi=300, bbox_inches='tight', transparent=True)

fig = sc.pl.embedding(adata_d3, basis='spatial', color='avg_distance_to_bronchi_zone',
                cmap = colormap_r,
                vmax=450,
                frameon=False, 
                size=50, 
                title = '',
                colorbar_loc = None,
                show=False, return_fig=True)
    
plt.xlim([3200, 3900])
plt.ylim([2900, 3600])
plt.savefig(os.path.join(plot_out_dir, "distance_day3_bronchi_zoom.png"), dpi=300, bbox_inches='tight', transparent=True)

## Plot 2D distances - ligand expression in all cells

In [ ]:
sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
adata_plot = sc.pp.subsample(adata_plot, n_obs=10000, copy=True)
for gene_name in ['Cxcl13', 'Ccl21a','Ccl19', 
                  'Tgfb1', 'Il33', 
                  'Cxcl16'
                 ]:
    p = distance_xy_kde(adata_plot,
                    sample_label=[sample_label],
                    x_axis = 'avg_distance_to_bronchi_zone',
                        # y_axis = 'avg_distance_to_TLS_zone',
                         y_axis = 'distance_to_tls',
                        gene_name = gene_name,
                    key_name = 'label_medium',
                    key_value = None,
                   x_clip = 450,
                   y_clip= 450,
                        vline = 150, hline = 100
                       )
    p
    plt.savefig(os.path.join(plot_out_dir, f"2D_{gene_name}_allcells_cyto.pdf"), dpi=300, bbox_inches='tight', transparent=True)


In [ ]:
# Run on larger number of cells - this will take awhile

import time

start = time.time()

sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
adata_plot = sc.pp.subsample(adata_plot, n_obs=50000, copy=True)
for gene_name in ['Ccl19','Il33',
                  # 'Cd274', 'Pdcd1lg2'
                 ]:
    p = distance_xy_kde(adata_plot,
                    sample_label=[sample_label],
                    x_axis = 'avg_distance_to_bronchi_zone',
                        # y_axis = 'avg_distance_to_TLS_zone',
                        y_axis = 'distance_to_tls',
                        gene_name = gene_name,
                    key_name = 'label_medium',
                    key_value = None,
                   x_clip = 450,
                   y_clip= 450,
                        vline = 150, hline = 100,
                        point_size = 2
                       )
    # p
    # p.savefig(os.path.join(plot_out_dir, f"2D_{gene_name}_allcells_cyto_50k_{sample_label}.pdf"), dpi=300, bbox_inches='tight', transparent=True)
    p.savefig(os.path.join(plot_out_dir, f"2D_{gene_name}_allcells_cyto_50k_{sample_label}.png"), dpi=300, bbox_inches='tight', transparent=True)


end = time.time()
duration = (end - start) / 60
print(f"{duration:.2f} minutes")

### Check which cells are expressing these ligands

In [ ]:
sc.pl.matrixplot(adata, var_names = ['Cxcl13', 'Ccl21a','Ccl19', 'Tgfb1', 'Il33'], groupby='label_fine', standard_scale = 'var', swap_axes=True)

In [ ]:
import session_info
print('active conda environment: ', os.path.basename(sys.prefix))
session_info.show()